In [3]:
# =========================================================
# MODIS AQUA LST → DISTRICT CSV
# =========================================================
#
# WORKFLOW
# --------
# 1. Load district GeoJSON
# 2. Convert to Earth Engine
# 3. Compute MODIS Aqua LST
# 4. Extract district mean LST
# 5. Append LST to district attributes
# 6. Remove geometry column
# 7. Export final CSV
#
# INPUT
# -----
# ../../assets/district.geojson
#
# OUTPUT
# ------
# ../../outputs/final_district_lst.csv
#
# =========================================================


# =========================================================
# 1. INSTALL PACKAGES
# =========================================================

%pip install earthengine-api geemap geopandas pandas


# =========================================================
# 2. IMPORT LIBRARIES
# =========================================================

import ee
import geemap
import geopandas as gpd
import pandas as pd

from pathlib import Path


# =========================================================
# 3. INITIALIZE EARTH ENGINE
# =========================================================
# Run ONCE in terminal:
#
# earthengine authenticate
#
# Then restart VS Code kernel

ee.Initialize()


# =========================================================
# 4. DEFINE PATHS
# =========================================================

geojson_path = Path('../../assets/district.geojson')

output_dir = Path('../../outputs')

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 5. CHECK FILE EXISTS
# =========================================================

print("GeoJSON exists:", geojson_path.exists())

if not geojson_path.exists():

    raise FileNotFoundError(
        f'GeoJSON not found: {geojson_path}'
    )


# =========================================================
# 6. LOAD GEOJSON
# =========================================================

gdf = gpd.read_file(geojson_path)

print("\nGeoJSON Loaded Successfully")


# =========================================================
# 7. OPTIONAL GEOMETRY SIMPLIFICATION
# =========================================================
# Helps reduce Earth Engine payload size

gdf['geometry'] = gdf.geometry.simplify(0.01)


# =========================================================
# 8. CONVERT TO EARTH ENGINE
# =========================================================

districts = geemap.geopandas_to_ee(gdf)

districts_simple = districts.map(
    lambda f: f.simplify(1000)
)

geom = districts_simple.geometry()


# =========================================================
# 9. DATE RANGE
# =========================================================

start = '2026-04-01'
end   = '2026-05-01'


# =========================================================
# 10. LOAD MODIS AQUA LST
# =========================================================
# Dataset:
# MODIS/061/MYD11A1

modis = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .filterBounds(geom)
    .filterDate(start, end)
)


# =========================================================
# 11. QA MASKING FUNCTION
# =========================================================

def mask_modis(img):

    qa = img.select('QC_Day')

    # Keep good + acceptable quality pixels
    mask = qa.bitwiseAnd(3).lte(1)

    return img.updateMask(mask)


# =========================================================
# 12. COMPUTE MONTHLY MEAN LST (°C)
# =========================================================

lst = (
    modis
    .map(mask_modis)
    .select('LST_Day_1km')
    .mean()
    .multiply(0.02)
    .subtract(273.15)
    .clip(geom)
)

print("\nLST Computed Successfully")


# =========================================================
# 13. DISTRICT-WISE MEAN LST
# =========================================================

district_stats = lst.reduceRegions(
    collection=districts_simple,
    reducer=ee.Reducer.mean(),
    scale=1000
)

print("\nDistrict Statistics Computed")


# =========================================================
# 14. CONVERT EE FEATURECOLLECTION TO PANDAS
# =========================================================

features = district_stats.getInfo()['features']

rows = []

for f in features:
    rows.append(f['properties'])

df = pd.DataFrame(rows)

print("\nDistrict DataFrame Created")


# =========================================================
# 15. RENAME MEAN COLUMN
# =========================================================

df = df.rename(
    columns={'mean': 'LST_C'}
)


# =========================================================
# 16. MERGE LST BACK TO ORIGINAL ATTRIBUTES
# =========================================================
#
# IMPORTANT:
# Replace 'dtname' if your district
# column name differs.
#

gdf_final = gdf.merge(
    df[['dtname', 'LST_C']],
    on='dtname',
    how='left'
)

print("\nLST Appended Successfully")


# =========================================================
# 17. DROP GEOMETRY COLUMN
# =========================================================

final_df = gdf_final.drop(
    columns='geometry'
)


# =========================================================
# 18. EXPORT FINAL CSV
# =========================================================

csv_output = output_dir / 'final_district_lst.csv'

final_df.to_csv(
    csv_output,
    index=False
)

print("\n===================================")
print("CSV EXPORTED SUCCESSFULLY")
print("===================================")

print(f"\nOutput File:\n{csv_output}")

Note: you may need to restart the kernel to use updated packages.
GeoJSON exists: True

GeoJSON Loaded Successfully

LST Computed Successfully

District Statistics Computed

District DataFrame Created

LST Appended Successfully

CSV EXPORTED SUCCESSFULLY

Output File:
../../outputs/final_district_lst.csv
